# 21장 seq2seq와 어텐션 — 어디를 얼마나 볼 것인가

**2단계 · 주석 달기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch21_2_comment.ipynb)

이 노트북에는 **주석을 지운 코드**가 들어 있습니다. 아래 표의 「주석 달기」 과제대로 내 말로 주석을 달아 보세요. 주석을 다는 동안 코드를 한 줄씩 다시 읽게 됩니다.

- 코드 셀 첫 줄의 `%%writefile 파일이름` 은 "이 셀의 내용을 그 파일로 저장하라"는 뜻입니다. 셀을 실행하면 내가 고친 내용이 파일에 저장됩니다.
- 그 아래 확인 셀을 실행해, 주석을 단 뒤에도 코드가 전과 똑같이 도는지 봅니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `p01_attention.py` 를 치고 실행. 첫 행을 종이로 계산해 대조 | [ch21_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch21_1_follow.ipynb) |
| 2. 주석 달기 | 세 단계(점수·가중치·가중 평균)의 shape을 각 줄에 | [ch21_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch21_2_comment.ipynb) **(지금 이 노트북)** |
| 3. 다시 만들기 | 주석만 남긴 파일에서 복원. 그다음 질의·키·값 선형층을 추가한 `Attention` 클래스로 만들기(22장 `MultiHeadAttention` 의 헤드 1개 판) | [ch21_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch21_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "21"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/21")

세 단계(점수·가중치·가중 평균)의 shape 을 각 줄에 주석으로 다세요.

## `p01_attention.py` — 대상: 파일 전체

In [ ]:
%%writefile p01_attention.py
import torch
import torch.nn.functional as F

torch.manual_seed(0)

tokens = ["원고는", "피고에게", "손해를", "배상하라고", "청구하였다"]
x = torch.randn(5, 4)

q = x[3]
scores = x @ q
print("점수      :", [round(s, 2) for s in scores.tolist()])

w = F.softmax(scores / x.shape[1] ** 0.5, dim=0)
print("가중치    :", {t: round(v, 2) for t, v in zip(tokens, w.tolist())}, "| 합", round(w.sum().item(), 2))

out = w @ x
print("출력 shape:", tuple(out.shape), "= 가중치로 섞은 벡터")

W = F.softmax((x @ x.T) / x.shape[1] ** 0.5, dim=1)
print("\n어텐션 행렬 [5,5] (행 합 = 1):")
for t, row in zip(tokens, W.tolist()):
    print(f"  {t:<6}", [round(v, 2) for v in row])
print("출력:", tuple((W @ x).shape), "= 입력과 같은 [5, 4], 각 토큰이 문맥을 섞어 넣은 새 벡터")

print("\n단어 주머니(평균)의 가중치:", [round(1 / 5, 2)] * 5)

### 확인

In [ ]:
!python p01_attention.py

### 원래대로 되돌리기

파일을 책의 원래 코드로 되돌리고 싶으면 아래 셀을 실행하세요. (Colab 에서 받은 저장소 기준입니다.)

In [ ]:
!git checkout -- p01_attention.py